## 1. Objetivo
Criar um DataFrame e aplicar transformações básicas:
- `select`
- `filter`
- `withColumn`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

spark = SparkSession.builder.appName("bloco1").getOrCreate()

In [ ]:
data = [
    (1, 'Ana', 21, 'São Paulo'),
    (2, 'Kaique', 32, 'Rio de Janeiro'),
    (3, 'Heloisa', 30, 'São Paulo'),
    (4, 'Carla', 40, 'Vila Velha'),
    (5, 'Yuri', 17, 'Osasco')
]

df = spark.createDataFrame(data, ['id', 'nome', 'idade', 'cidade'])

df_result = (
    df.select('nome', 'idade')
    .filter(col('idade') > 25)
    .withColumn('maior_de_idade', when(col('idade') >= 18, True).otherwise(False))
)

df_result.show()

## 2. Objetivo:
- Usar um dataset simples (lista de números ou nomes)
- Aplicar transformações básicas

**Atividades:**
- **map** → dobrar valores  
- **filter** → manter apenas números pares  

**Conceito:**
- **Narrow Transformation** → não precisa de shuffle  
- Cada partição é processada de forma independente  

In [ ]:
rdd = spark.sparkContext.parallelize([1,2,3,4,5,6])

result = (
    rdd.map(lambda x: x * 2)
    .filter(lambda x: x % 2 == 0)
)

print(result.collect())

## 3. Objetivo:
- Trabalhar com um DataFrame
- Criar novas colunas utilizando `withColumn`
- Encadear múltiplas transformações

**Atividades:**
- Criar 3 novas colunas com `withColumn`
- Encadear as operações em sequência

**Insight:**
- Todas as operações continuam sendo **narrow transformations**
- Não há troca de dados entre partições (sem shuffle)

In [ ]:
from pyspark.sql.functions import lit

df2 = (
    df.withColumn('idade_dobro', col('idade') * 2)
    .withColumn('pais', lit('Brasil'))
    .withColumn('idade_mais_10', col('idade') + 10)
)

df2.show()

## 4. Objetivo
- Trabalhar com agregações em um DataFrame
- Entender o comportamento de `groupBy`

**Dataset:**
- Colunas: `categoria`, `valor`

**Atividades:**
- Agrupar por `categoria`
- Calcular a soma dos valores por categoria

**Insight:**
- **Wide Transformation** → ocorre **shuffle**
- Os dados são redistribuídos entre as partições para realizar a agregação

In [ ]:
data = [
    ('eletronico', 100),
    ('eletronico', 200),
    ('roupa', 50),
    ('roupa', 80)
]

df = spark.createDataFrame(data, ['categoria', 'valor'])

df_grouped = df.groupBy('categoria').sum('valor')

df_grouped.show()

## 5. Objetivo:
- Trabalhar com remoção de duplicatas em um DataFrame
- Comparar os métodos `distinct()` e `dropDuplicates()`

**Atividades:**
- Criar um dataset com dados duplicados
- Aplicar `.distinct()`
- Aplicar `.dropDuplicates()`
- Comparar os resultados

**Pergunta:**
- Por que essas operações geram **shuffle**?

In [ ]:
data = [('São Paulo',), ('Rio de Janeiro',), ('São Paulo',), ('Vila Velha',), ('Osasco',)]
df = spark.createDataFrame(data, ['cidade'])

df.distinct().show()
df.dropDuplicates().show()
df.show()

## 6. Objetivo:
- Trabalhar com junção de DataFrames
- Entender o impacto de performance dos joins

**Dataset:**
- **DF1:** `id`, `nome`  
- **DF2:** `id`, `salario`  

**Atividades:**
- Realizar join entre os DataFrames utilizando a coluna `id`

**Insight importante:**
- **Join = Wide Transformation**
- Envolve **shuffle** (redistribuição dos dados entre partições)
- Dependendo do tipo de join → pode ter **custo alto de processamento**

In [ ]:
df1 = spark.createDataFrame([
    (1, "Ana"),
    (2, "Bruno")
], ["id", "nome"])

df2 = spark.createDataFrame([
    (1, 3000),
    (2, 4000)
], ["id", "salario"])

df_join = df1.join(df2, on="id", how="inner")

df_join.show()

## 7. Objetivo:
- Entender a diferença entre **narrow** e **wide transformations** em um fluxo completo
- Identificar onde ocorre **shuffle**

**Pipeline:**
- `filter` → narrow  
- `withColumn` → narrow  
- `groupBy` → wide  
- `select` → narrow  

**Atividades:**
- Executar o pipeline completo
- Identificar em qual etapa ocorre o **shuffle**
- Explicar o motivo

**Insight:**
- O **shuffle** acontece no `groupBy`, pois exige redistribuição dos dados entre partições para realizar a agregação

In [ ]:
from pyspark.sql.functions import avg

data = [
    ("eletronico", 100),
    ("eletronico", 200),
    ("roupa", 50),
]

df = spark.createDataFrame(data, ["categoria", "valor"])

pipeline = (
    df.filter(col("valor") > 50)
      .withColumn("valor2", col("valor") * 2)
      .groupBy("categoria").agg(avg("valor2"))
      .select("categoria", "avg(valor2)")
)

pipeline.show()

## 8. Objetivo:
- Entender como funciona a redistribuição de dados em Spark
- Comparar `repartition` e `coalesce`

**Atividades:**
- Criar um DataFrame grande
- Aplicar `.repartition(10)`
- Aplicar `.coalesce(2)`
- Observar diferenças de execução e performance

**Insight:**
- **repartition** → **wide transformation** (gera shuffle)
- **coalesce** → tenta evitar shuffle (mais eficiente quando reduz partições)

In [ ]:
df = spark.range(0, 1000)

df_rep = df.repartition(10)
df_coa = df.coalesce(2)

print(df_rep.rdd.getNumPartitions())
print(df_coa.rdd.getNumPartitions())

## 9. Objetivo:
- Entender o uso de cache em Spark para otimização de performance
- Evitar reprocessamento de transformações custosas

**Atividades:**
- Criar um DataFrame com transformação pesada
- Aplicar `.cache()` (ou `.persist()`)
- Executar duas ações (ex: `count()` e `show()`)
- Comparar o tempo de execução

**Insight:**
- Sem cache → o Spark recalcula tudo a cada ação
- Com cache → os dados ficam armazenados em memória, evitando reprocessamento
- Resultado: melhora significativa de performance em múltiplas ações

In [ ]:
df = spark.range(0, 1000000).withColumn("x2", col("id") * 2)

df.cache()

df.count()
df.count()

## 10. Objetivo:
- Entender o plano de execução do Spark
- Identificar operações de **shuffle** e estágios (stages)

**Atividades:**
- Executar `df.explain()`
- Analisar o plano gerado

**Tente identificar:**
- Onde aparece **Exchange** (indica shuffle)
- Como os **stages** estão organizados

In [ ]:
df = spark.range(0, 1000)

df_filtered = df.filter(col("id") > 10)

df_filtered.explain(True)

## 11. Objetivo:
- Simular um cenário real de análise de dados
- Aplicar transformações e agregações em conjunto

**Dataset:**
- `id_pedido`  
- `produto`  
- `categoria`  
- `valor`  

**Atividades:**
- Calcular faturamento por categoria (`groupBy`)
- Identificar os top produtos
- Calcular o ticket médio
- Filtrar os dados dos últimos 3 meses

**Desafio:**
- Identificar quais operações são **wide transformations**
- Explicar onde ocorre **shuffle** e por quê

In [ ]:
from pyspark.sql.functions import sum, avg

data = [
    (1, "notebook", "eletronico", 3000),
    (2, "celular", "eletronico", 2000),
    (3, "camiseta", "roupa", 100),
    (4, "calça", "roupa", 150),
]

df = spark.createDataFrame(data, ["id_pedido", "produto", "categoria", "valor"])

faturamento = df.groupBy("categoria").agg(sum("valor").alias("total"))
ticket_medio = df.agg(avg("valor").alias("ticket_medio"))

faturamento.show()
ticket_medio.show()

faturamento.explain(True)
ticket_medio.explain(True)